In [ ]:
!pip install -q -U google-genai

In [ ]:
!pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 20.9 MB/s eta 0:00:00


In [ ]:
from google import genai

client = genai.Client(api_key="**")

response = client.models.generate_content(
    model="gemini-2.0-flash",
    contents="explain different type of stars in few words"
)
print(response.text)

Okay, here's a breakdown of different star types in a few words each:

*   **Main Sequence Stars:** Normal, hydrogen-burning stars (like our Sun).

*   **Red Giants:** Large, cooler, evolved stars nearing the end of their life.

*   **White Dwarfs:** Small, dense, hot remnants of dead stars.

*   **Neutron Stars:** Extremely dense, rapidly spinning remnants after a supernova.

*   **Black Holes:** Extremely dense objects with gravity so strong nothing escapes.

*   **Red Dwarfs:** Small, cool, long-lived, very common stars.

*   **Blue Giants/Supergiants:** Massive, hot, luminous, short-lived stars.



In [ ]:
chat = client.chats.create(model="gemini-2.0-flash")
response = chat.send_message("I have 2 dogs in my house.")
print(response.text)

response = chat.send_message("How many paws are in my house?")
print(response.text)

for message in chat.get_history():
    print(f'role - {message.role}',end=": ")
    print(message.parts[0].text)

Okay! Two dogs can definitely make a house lively. Do you want to tell me anything else about them? Like their names, breeds, or what they like to do?

Since you have two dogs, and each dog has four paws, you have 8 paws in your house (2 dogs x 4 paws/dog = 8 paws).

role - user: I have 2 dogs in my house.
role - model: Okay! Two dogs can definitely make a house lively. Do you want to tell me anything else about them? Like their names, breeds, or what they like to do?

role - user: How many paws are in my house?
role - model: Since you have two dogs, and each dog has four paws, you have 8 paws in your house (2 dogs x 4 paws/dog = 8 paws).



In [ ]:
import fitz  # PyMuPDF
from google.colab import files
import io
from IPython.display import display
import ipywidgets as widgets

# Widget for uploading PDF
upload_widget = widgets.FileUpload(accept='.pdf', multiple=False)
display(upload_widget)

# Read the uploaded file into PyMuPDF
if upload_widget.value:
    file_info = next(iter(upload_widget.value.values()))

FileUpload(value={}, accept='.pdf', description='Upload')

In [ ]:
print("📎 Upload your hospital or neural report PDF:")
uploaded = files.upload()

API_KEY = "**"

# Extract text from PDF
def extract_pdf_text(file_bytes):
    with fitz.open(stream=file_bytes, filetype="pdf") as doc:
        text = ""
        for i, page in enumerate(doc):
            text += f"\n--- Page {i + 1} ---\n" + page.get_text()
    return text

# 4. Gemini to answer prompt
def query_gemini(pdf_text, user_question):
    client = genai.Client(api_key=API_KEY)
    response = client.models.generate_content(
        model=  "gemini-2.0-flash",
        contents=f"""
You're a helpful assistant reading a medical/neuroscience report.

PDF Content:
{pdf_text}

Question: {user_question}

Give a concise answer and refer to page numbers when relevant.
"""
    )
    return response.text

# 5. Get PDF file and user prompt
filename = list(uploaded.keys())[0]
pdf_bytes = uploaded[filename]
pdf_text = extract_pdf_text(io.BytesIO(pdf_bytes))

# 6. Ask your question
print("\n❓ Enter your question about the document:")
user_prompt = input()

# 7. Generate and display the answer
answer = query_gemini(pdf_text, user_prompt)
print("\n✅ Gemini's Answer:\n")
print(answer)

📎 Upload your hospital or neural report PDF:


Saving mri.pdf to mri.pdf

❓ Enter your question about the document:
problem 

✅ Gemini's Answer:

Munchkin Malost, a Himalayan cat, is experiencing seizures and has been diagnosed with a feline form of craniosynostosis, leading to several brain and spinal cord abnormalities including syringomyelia. (Page 1). The prognosis for remission is guarded due to the severity of these congenital/developmental issues (Page 1).



In [ ]:
from google import genai
import fitz  # PyMuPDF
from google.colab import files
import io

#  Upload PDF
print("📄 Upload your hospital/neural report PDF")
uploaded = files.upload()
filename = list(uploaded.keys())[0]
pdf_bytes = uploaded[filename]

# Extract Text from PDF
def extract_pdf_text(file_bytes):
    with fitz.open(stream=file_bytes, filetype="pdf") as doc:
        text = ""
        for i, page in enumerate(doc):
            text += f"\n--- Page {i+1} ---\n" + page.get_text()
    return text

pdf_text = extract_pdf_text(io.BytesIO(pdf_bytes))

# Gemini API Key
import getpass
API_KEY = getpass.getpass("🔑 Enter your Gemini API Key: ")
client = genai.Client(api_key=API_KEY)

# Multiple Questions
def ask_gemini(pdf_text, user_prompt):
    response = client.models.generate_content(
        model="gemini-1.5-pro",
        contents=f"""
You are a helpful assistant reading a medical or neuroscience report.

PDF Content:
{pdf_text}

User's Question: "{user_prompt}"

Respond clearly and reference page numbers if relevant.
"""
    )
    return response.text

# loop
while True:
    user_input = input("\n❓ Ask your question (or type 'exit' to quit): ")
    if user_input.strip().lower() in ['exit', 'quit']:
        print("👋 Ending session.")
        break
    answer = ask_gemini(pdf_text, user_input)
    print("\n✅ Gemini's Answer:\n")
    print(answer)
